**Perubahan 1 dari 5: bagian import**

In [ ]:
'''HAPUS'''

from langchain_huggingface import HuggingFaceEmbeddings


Kenapa dihapus duluan: baris ini yang tadinya "mendatangkan" petugas embedding dari luar (HuggingFace). Karena nanti kita ganti dengan petugas bawaan Chroma, baris impor ini sudah tidak dipakai lagi di manapun dalam file ini.

---

**Perubahan 2 dari 5: dua baris konfigurasi**

In [ ]:
'''HAPUS'''

EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
PERSIST_DIR = "./chroma_db"

EMBEDDING_MODEL dulu berfungsi sebagai "alamat" nama model yang diberikan ke HuggingFaceEmbeddings supaya dia tahu model mana yang harus diunduh. Sekarang, karena kita pakai petugas bawaan Chroma, kita tidak perlu memberi tahu nama modelnya sama sekali. Chroma sudah otomatis tahu memakai all-MiniLM-L6-v2 sendiri tanpa perlu disebut di kode kita. Jadi variabel ini kehilangan fungsi sepenuhnya, bukan cuma "dipindah pemakaiannya", tapi memang sudah tidak relevan.

PERSIST_DIR ini agak berbeda ceritanya, dan ini penting dipahami supaya Bapak bisa jelaskan ke peserta dengan percaya diri. Variabel ini dulu menunjuk ke folder chroma_db, tempat menyimpan arsip vector store secara permanen di disk, supaya kalau program dimatikan lalu dinyalakan lagi, arsipnya tidak hilang dan tidak perlu dibangun ulang dari nol.

Tapi kalau Bapak perhatikan lagi ke fungsi bangun_vectorstore() (yang akan kita ubah di Perubahan 4), di situ ada baris vectorstore.reset_collection() yang dipanggil setiap kali program jalan. Artinya, walaupun kita menyimpan arsip secara permanen ke folder chroma_db, arsip itu langsung dikosongkan dan disusun ulang dari awal setiap kali main() dipanggil. Jadi menyimpannya secara permanen itu sia-sia, seperti menyewa gudang tapi setiap pagi gudangnya dibongkar habis dan diisi ulang dari nol. Untuk kasus penggunaan kita (dokumen sumbernya tetap, tidak pernah ditambah user secara live), penyimpanan permanen ini tidak memberi manfaat apapun, hanya menambah kompleksitas.

---

**Perubahan 3 dari 5: fungsi buat_embeddings()**

In [ ]:
'''HAPUS'''
def buat_embeddings() -> HuggingFaceEmbeddings:
    """Siapkan model embedding, jalan lokal tanpa API key tambahan."""
    return HuggingFaceEmbeddings(model_name=EMBEDDING_MODEL)

Fungsi buat_embeddings() ini tugasnya cuma satu: memanggil HuggingFaceEmbeddings dengan nama model dari EMBEDDING_MODEL, lalu menyerahkan hasilnya (objek petugas embedding) ke pemanggilnya. Sekarang dua bahan yang dia butuhkan sudah tidak ada, HuggingFaceEmbeddings sudah tidak diimpor (Perubahan 1), dan EMBEDDING_MODEL sudah tidak ada (Perubahan 2). Jadi fungsi ini otomatis tidak bisa jalan lagi kalau dibiarkan, harus dihapus.

---

**Perubahan 4 dari 5: fungsi bangun_vectorstore()**

- Cari section DATA INGESTION
- Cari fungsi `bangun_vectorstore()`

4a

In [ ]:
'''UBAH DARI'''
def bangun_vectorstore(dokumen: list, embeddings: HuggingFaceEmbeddings) -> Chroma:

'''JADI'''
def bangun_vectorstore(dokumen: list) -> Chroma:

Sekarang bahan embeddings sudah tidak ada yang mengirim (karena buat_embeddings() sudah dihapus di Perubahan 3), jadi parameter ini harus ikut dibuang dari definisi fungsi. Kalau dibiarkan, nanti Python akan komplain "missing required argument" begitu fungsi ini dipanggil tanpa bahan kedua.

4b

In [ ]:
'''CARI'''
vectorstore = Chroma(
    collection_name=COLLECTION_NAME,
    embedding_function=embeddings,
    persist_directory=PERSIST_DIR,
)

'''UBAH JADI'''
# Tanpa embedding_function, Chroma otomatis memakai embedder bawaannya
# sendiri (model ONNX all-MiniLM-L6-v2, sama persis dengan yang dipakai
# sebelumnya lewat HuggingFace). Tanpa persist_directory, data disimpan
# di memori saja, bukan di folder chroma_db. Ini tidak masalah karena
# reset_collection() di bawah selalu menyusun ulang datanya dari nol
# setiap kali program ini dijalankan.
vectorstore = Chroma(
    collection_name=COLLECTION_NAME,
)

Sekarang, kalau embedding_function tidak diisi sama sekali, Chroma tidak akan error atau kosong. Dia otomatis memanggil petugas bawaannya sendiri, yang disebut ONNX embedder, dan modelnya kebetulan model yang sama persis, all-MiniLM-L6-v2. Ini penting ditegaskan ke peserta: hasil akhirnya sama, cuma cara mendapatkan petugasnya yang beda. Bedanya, petugas bawaan Chroma ini "ringan", jalan di atas onnxruntime yang sudah otomatis ikut terpasang bersama chromadb, tidak perlu unduhan tambahan sebesar torch.

---

**Perubahan 5 dari 5: fungsi main()**

5a

In [ ]:
'''CARI'''
print("Menyiapkan model dan embedding...")
model = buat_model()
embeddings = buat_embeddings()

'''UBAH JADI'''
print("Menyiapkan model...")
model = buat_model()

Baris embeddings = buat_embeddings() ini memanggil fungsi yang sudah kita hapus di Perubahan 3. Kalau baris ini dibiarkan, begitu program dijalankan akan langsung muncul error NameError: name 'buat_embeddings' is not defined, karena Python mencari fungsi dengan nama itu dan tidak menemukannya sama sekali. Teks print-nya saya minta disesuaikan supaya tetap jujur menggambarkan apa yang sebenarnya terjadi, karena sekarang di baris itu memang cuma model yang disiapkan.

5b

In [ ]:
'''CARI'''
print("Membangun vector store dari dokumen...")
vectorstore = bangun_vectorstore(dokumen, embeddings)

'''UBAH JADI'''
print("Membangun vector store dari dokumen...")
vectorstore = bangun_vectorstore(dokumen)

Ini konsekuensi langsung dari Langkah 4a tadi, di mana kita sudah mengubah definisi fungsi bangun_vectorstore() supaya cuma menerima satu bahan, yaitu dokumen. Kalau baris pemanggilan ini tetap mengirim dua bahan (dokumen, embeddings), Python akan komplain TypeError: bangun_vectorstore() takes 1 positional argument but 2 were given. Variabel embeddings sendiri sebenarnya sudah tidak ada lagi sejak Langkah 5a, jadi kalaupun tidak diubah, errornya justru akan muncul lebih dulu di sini sebagai NameError karena variabelnya tidak dikenal.